In [ ]:
import pandas as pd
import re
import numpy as np

# Pliki wejściowe
files = [
    "europeana_articles.csv", 
    "europeana_articles_2turn.csv", 
    "europeana_articles_3turn.csv"
]
output_file = "europeana_articles_fixed.csv"

# Listy konfiguracyjne

# 1. Wyrażenia do usunięcia z treści (zamiana na pusty string)
phrases_to_remove = [
    "Numérisation effectuée à partir d'un document original.",
    "Numérisation effectuée à partir d'un document de substitution.",
    "Scanning from a substitute document.",
    "Scanning from an original document.",
    "Bible with the Glossa Ordinaria",
    "Copia digital.",
    "Scanned by the partner",
    "With text mode",
    "Title page missing",
    "Calendarium Jaurinense titulare historicum ad annum Jesu Christi",
    "#agentOf:",
    "Chronica, monthly magazine of the Central Jewish Council, issue n.",
    "El Porvenir: No.",
    "Biologisches Centralblatt",
    "Commentarii de rebvs in scientia natvrali et medicina gestis",
    "[---] / [---]", "[---] /", "[---]", "〚[---]〛", "[------]", "[ - - - - - - - - - - ]",
    "Text page with plain ink initials", "Text page.", "Start of text."
]

# 2. Wyrażenia dyskwalifikujące cały wiersz (jeśli wystąpią w title/creator/abstract)
rows_to_drop_phrases = [
    "Commentarii de rebus in scientia naturali et medicina gestis",
    "Commentarii perpetvi in classicos romanorvm scriptores",
    "COMMENTARII TRES in Vniversam THEOLOGIAM Scolasticam",
    "City of Wrocław Magistrate",
    "Czasopismo naukowe założone w 2001 roku",
    "Słownik łaciny średniowiecznej w Polsce",
    "Dictionary of medieval Latin in Poland",
    "Acta Scientiarum Polonorum. Medicina Veterinaria",
    "Acta Scientiarum Polonorum. Geodesia",
    "Acta Scientiarum Polonorum. Biotechnology",
    "organ of the Polish Philological Society",
    "organ Polskiego Towarzystwa Filologicznego",
    "Gazeta Kościerska",
    "Schlesisches Bonifatius-Vereins-Blatt."
]

# Funkcje

def clean_europeana_text(text):
    """
    1. Usuwa frazy z listy phrases_to_remove.
    2. Spłaszcza tekst.
    3. Usuwa znane tagi HTML (<p>, <em>), a w innych (np. <Anonymus>) usuwa tylko klamry <>.
    """
    if not isinstance(text, str):
        return ""
    
    # A. Usuwanie konkretnych fraz
    for phrase in phrases_to_remove:
        if phrase in text:
            text = text.replace(phrase, "")

    # B. Spłaszczanie
    text = re.sub(r'\s+', ' ', text).strip()
    
    # C. Usuwanie znanych tagów HTML (całych)
    text = re.sub(r'</?(p|em|br|b|i|strong|span|div)\s*/?>', ' ', text, flags=re.IGNORECASE)
    
    # D. Usuwanie samych znaków < i > (zostawiając treść, np. <1585> -> 1585)
    text = re.sub(r'[<>]', '', text)
    
    return text.strip()

def extract_last_year(val):
    """
    Pobiera ostatni 4-cyfrowy rok z ciągu.
    Dla '1728; 1959' -> 1959.
    Dla '14900906' -> 1490 (pierwsze 4 cyfry).
    """
    if pd.isna(val):
        return np.nan
    val = str(val)
    
    ## szuka wszystkich potencjalnych lat (1000-2029)
    ## Regex łapie 4 cyfry, które są samodzielne LUB są początkiem długiego ciągu (dla 14900906)
    matches = re.findall(r'(1\d{3}|20[0-2]\d)', val)
    
    if matches:
        ## bierze ostatni znaleziony rok (wg instrukcji "1728; 1716; 1959" -> 1959)
        ## przy "14900906" znajdzie 1490.
        return int(matches[-1])
        
    return np.nan

def is_short_content(row):
    """
    Sprawdza warunek: Title <= 30 znaków ORAZ Abstract <= 30 znaków (bez interpunkcji) ORAZ brak creatora.
    Zwraca True, jeśli wiersz jest "zbyt krótki/śmieciowy".
    """
    creator = str(row.get('creator', '')).strip()
    if creator and creator.lower() != 'nan':
        return False # jest autor więc zostawia
    
    # Funkcja licząca tylko litery i cyfry
    def count_chars(s):
        if not isinstance(s, str): return 0
        return len(re.sub(r'[\W_]+', '', s)) # usuwa spacje i interpunkcję
    
    t_len = count_chars(row.get('title', ''))
    a_len = count_chars(row.get('abstract', ''))
    
    # Jeśli tytuł krótki I abstrakt krótki (i brak autora), to kosz
    return t_len <= 30 and a_len <= 30


# Główny proces

print("1. Wczytywanie i łączenie plików...")
try:
    df_list = [pd.read_csv(f, on_bad_lines='skip', dtype=str) for f in files]
    df = pd.concat(df_list, ignore_index=True)
    print(f"Połączono. Łącznie wierszy na start: {len(df):,}")
except Exception as e:
    print(f"Błąd krytyczny: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 2. Wstępne czyszczenie tekstów (Title, Creator, Abstract)
    print("2. Czyszczenie tekstów (tagi, frazy, spłaszczanie)...")
    cols_to_clean = ['title', 'abstract', 'creator', 'subject', 'publisher']
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = df[col].astype(str).replace('nan', '').apply(clean_europeana_text)

    # 3. FILTROWANIE ŚMIECI (WIERSZE DO USUNIĘCIA)
    print("3. Filtrowanie wierszy (logika biznesowa)...")
    initial_count = len(df)

    # A. Usuwanie po frazach (tytuł, autor, abstrakt)
    # Łączymy kolumny do sprawdzenia
    content_check = df['title'] + " " + df['creator'] + " " + df['abstract']
    mask_phrases = content_check.apply(lambda x: any(bad in x for bad in rows_to_drop_phrases))
    
    # B. Usuwanie "a.s.a." w creator
    mask_asa = df['creator'].str.contains("a.s.a.", case=False, regex=False)
    
    # C. Usuwanie pustych (brak creator + abstract + year jednocześnie)
    # najpierw musi wyliczyć year, żeby to sprawdzić, ale year jest brudny.
    # sprawdź 'year' surowy na obecność jakiejkolwiek treści
    mask_empty_cay = (
        (df['creator'].str.len() < 2) & 
        (df['abstract'].str.len() < 2) & 
        (df['year'].astype(str).str.len() < 2) # sprawdza czy w ogóle coś tam jest
    )

    # D. Usuwanie krótkich (title <= 30 AND abstract <= 30 AND no creator)
    # to jest wolniejsze, robi apply na wierszach
    mask_short = df.apply(is_short_content, axis=1)

    # Łączymy wszystkie warunki usunięcia (OR)
    rows_to_drop = mask_phrases | mask_asa | mask_empty_cay | mask_short
    
    df = df[~rows_to_drop]
    print(f"Usunięto {initial_count - len(df):,} rekordów (filtry 'a.s.a.', 'krótkie', 'zakazane frazy', 'puste').")

    # 4. Naprawa daty
    print("4. Formatowanie daty (ostatnie 4 cyfry)...")
    if 'year' in df.columns:
        df['year'] = df['year'].apply(extract_last_year).astype('Int64')

    # 5. Deduplikacja
    dedup_cols = ['title', 'creator']
    print(f"5. Deduplikacja wg: {dedup_cols}...")
    before_dedup = len(df)
    # Przed deduplikacją warto zamienić puste stringi na NaN w creator, żeby deduplikacja działała sensownie
    # (ale w tym zbiorze puste creatory są częste, więc zostawia puste stringi jako wartość)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    print(f"Usunięto duplikatów: {before_dedup - len(df):,}")

    # 6. Zapis
    print(f"6. Zapisywanie do {output_file}...")
    # wybór kolumn (year idzie na float/Int64 więc jest bezpieczny)
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # Podgląd
    print("\n--- Wynik ---")
    print(df[['title', 'year']].head(5))
    print(f"Liczba końcowa rekordów: {len(df):,}")
